In [1]:
import requests
import time 

In [2]:
params = {
    "page":1,
    "per_page": 100,
    "strategy": 'new'
}

API = "https://www.tabnews.com.br/api/v1"

In [3]:
errors_report = []

In [4]:
def request_with_retry(url, params=None, max_retries=3):
    """Faz a requisição e, se der 429, aplica espera exponencial e tenta de novo."""
    for tentativa in range(max_retries):
        response = requests.get(url, params=params)
        
        if response.status_code == 200:
            return response
        
        elif response.status_code == 429:
            # Pausa progressiva curta: 1.5s na 1ª tentativa, 3s na 2ª, 4.5s na 3ª
            tempo_espera = (tentativa + 1) * 1.5
            print(f"⚠️ Rate limit (429) em {url}. Tentativa {tentativa+1}/{max_retries}. Pausando {tempo_espera}s...")
            time.sleep(tempo_espera)
        else:
            # Se for outro erro (ex: 404, 500), sai do loop e retorna a resposta com erro
            break
            
    return response

def scrape_page(current_params):
    response = request_with_retry(f'{API}/contents', params=current_params)

    if response.status_code == 200:
        return response.json()
    else:
        error = f"Erro na página {current_params['page']}: {response.status_code}"
        print(error)
        errors_report.append(error)
        return []


def scrape_site(num_pages):
    all_content = []
    for i in range(num_pages):
        params["page"] = i+1
        page_content = scrape_page(params)

        if page_content:
            all_content.extend(page_content)
        else:
            break

    return all_content

def get_post_body(user, slug):
    response = request_with_retry(f'{API}/contents/{user}/{slug}')

    if response.status_code == 200:
        return response.json()
    else:
        error = f"Erro ao coletar dados do post {slug}: {response.status_code}"
        print(error)
        errors_report.append(error)
        return []

def get_post_comments(user, slug):
    response = request_with_retry(f'{API}/contents/{user}/{slug}/children')

    if response.status_code == 200:
        return response.json()
    else:
        error = f"Erro ao coletar comentários do post {slug}: {response.status_code}"
        print(error)
        errors_report.append(error)
        return []


In [5]:
raw_data_inc = scrape_site(50)
print(f'Quantidade de material coletado {len(raw_data_inc)}')

Quantidade de material coletado 5000


In [ ]:
raw_data = []
comments_data = []
for post in raw_data_inc:
    body = get_post_body(post["owner_username"], post["slug"])
    comments = get_post_comments(post["owner_username"], post["slug"])
    comments_data.append(comments)
    raw_data.append(body)


⚠️ Rate limit (429) em https://www.tabnews.com.br/api/v1/contents/jeansantos2020/vale-a-pena-integrar-open-finance-num-side-project-ou-cadastro-manual-ja-resolve-pra-validar. Tentativa 1/3. Pausando 1.5s...
⚠️ Rate limit (429) em https://www.tabnews.com.br/api/v1/contents/jeansantos2020/vale-a-pena-integrar-open-finance-num-side-project-ou-cadastro-manual-ja-resolve-pra-validar. Tentativa 2/3. Pausando 3.0s...
⚠️ Rate limit (429) em https://www.tabnews.com.br/api/v1/contents/whyrnld/criei-uma-skill-open-source-para-integrar-a-api-do-banco-inter-usando-chatgpt-codex-e-claude-code. Tentativa 1/3. Pausando 1.5s...
⚠️ Rate limit (429) em https://www.tabnews.com.br/api/v1/contents/Jemorini/falha-de-xss-em-sistema-de-ingressos. Tentativa 1/3. Pausando 1.5s...
⚠️ Rate limit (429) em https://www.tabnews.com.br/api/v1/contents/Jemorini/falha-de-xss-em-sistema-de-ingressos/children. Tentativa 1/3. Pausando 1.5s...
⚠️ Rate limit (429) em https://www.tabnews.com.br/api/v1/contents/NewsletterOficia

In [ ]:
len(raw_data)

In [ ]:
users = []
posts = []
comments = []

for post_item in raw_data:
    if isinstance(post_item, dict):
        user = {
            "user_id": post_item.get("owner_id"),
            "username": post_item.get("owner_username")
        }

        post = {
            "id": post_item.get("id"),
            "title": post_item.get("title"),
            "body": post_item.get("body"),
            "created_at": post_item.get("created_at"),
            "updated_at": post_item.get("updated_at"),
            "published_at": post_item.get("published_at"),
            "user_id": post_item.get("owner_id"),
            "tabcoins": post_item.get("tabcoins")
        }

        users.append(user)
        posts.append(post)
    else:
        continue

for comment_list in comments_data:
    if isinstance(comment_list, list):
        for comment_info in comment_list:
            if isinstance(comment_info, dict):
                comment = {
                    "id": comment_info.get("id"),
                    "user_id": comment_info.get("owner_id"),
                    "post_id": comment_info.get("parent_id"),
                    "body": comment_info.get("body")
                }
                comments.append(comment)
            else:
                continue
    else:
        continue


In [ ]:
import pandas as pd

In [ ]:
users_df = pd.DataFrame(users)
posts_df = pd.DataFrame(posts)
comments_df = pd.DataFrame(comments)

In [ ]:
# Usuários
total_users_before = len(users_df)
users_df = users_df.drop_duplicates(subset=['user_id'])
print(f"👥 Usuários: de {total_users_before} para {len(users_df)} (Removidos: {total_users_before - len(users_df)})")

# Posts
total_posts_before = len(posts_df)
posts_df = posts_df.drop_duplicates(subset=['id'])
print(f"📝 Posts: de {total_posts_before} para {len(posts_df)} (Removidos: {total_posts_before - len(posts_df)})")

# Comentários
total_comments_before = len(comments_df)
comments_df = comments_df.drop_duplicates(subset=['id'])
print(f"💬 Comentários: de {total_comments_before} para {len(comments_df)} (Removidos: {total_comments_before - len(comments_df)})")

In [ ]:
posts_df

In [ ]:
posts

In [ ]:
data_path = '../csv_data/'

users_df.to_csv(f'{data_path}users_data.csv', index=False, encoding='utf-8')
posts_df.to_csv(f'{data_path}posts_data.csv', index=False, encoding='utf-8')
comments_df.to_csv(f'{data_path}comments_data.csv', index=False, encoding='utf-8')

In [ ]:
from datetime import datetime

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

with open(f'../scrapping_errors/scrapping-errors-report-{timestamp}.txt', 'w', encoding='utf-8') as f:
    for line in errors_report:
        f.write(f"{line}\n")


In [ ]:
len(posts_df)